# Week 6.5: Neural Networks and Model Choice

This guided lab uses three public datasets downloaded from Kaggle. For each one, you will:

1. Print the dataset and inspect its columns.
2. Name the **target** you want to predict.
3. Decide between **regression** and **classification**.
4. Choose features that would be available before the prediction.
5. Train a small neural network and interpret one useful score.

The setup code is provided. Your main job is to make and explain the modeling choices.


## Quick review

A neural network receives **features** as inputs and produces a prediction. The target determines the type of prediction:

- **Regression** predicts a measured number, such as rent or temperature.
- **Classification** predicts a category, such as Yes/No or one of several teams.

A useful feature should exist before the prediction and should not simply reveal the answer. IDs usually identify rows rather than explain the target. Information created after the outcome can cause **data leakage**.


## 1. Import the tools

Run this cell once. It installs KaggleHub when needed, then imports pandas and scikit-learn. KaggleHub downloads public Kaggle datasets directly into the notebook environment.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 25)


## 2. Download three Kaggle datasets

The next cell downloads three small CSV datasets:

- [House Rent Prediction Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/house-rent-prediction-dataset)
- [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)
- [Iris Species](https://www.kaggle.com/datasets/uciml/iris)

KaggleHub caches each download, so rerunning the cell usually reuses the local copy. Run the cell and confirm that all three shapes print successfully.


In [ ]:
def find_file(download_dir, filename):
    matches = list(Path(download_dir).rglob(filename))
    if not matches:
        available = [path.name for path in Path(download_dir).rglob("*.csv")]
        raise FileNotFoundError(f"Could not find {filename}. CSV files found: {available}")
    return matches[0]


rent_dir = kagglehub.dataset_download("iamsouravbanerjee/house-rent-prediction-dataset")
churn_dir = kagglehub.dataset_download("blastchar/telco-customer-churn")
iris_dir = kagglehub.dataset_download("uciml/iris")

apartment_data = pd.read_csv(find_file(rent_dir, "House_Rent_Dataset.csv"))
renewal_data = pd.read_csv(find_file(churn_dir, "WA_Fn-UseC_-Telco-Customer-Churn.csv"))
iris_data = pd.read_csv(find_file(iris_dir, "Iris.csv"))

print("Datasets downloaded from Kaggle:")
print("apartment_data:", apartment_data.shape)
print("renewal_data:", renewal_data.shape)
print("iris_data:", iris_data.shape)


## 3. The guided training helper

This function handles the repetitive setup after you choose the target, task type, and features. It will:

- split rows into training and test sets,
- fill missing values,
- scale numeric features,
- one-hot encode categorical features,
- train either an MLP regressor or MLP classifier,
- print a score that fits the task.

Read the function if you are curious, but you do not need to edit it.


In [ ]:
def run_guided_model(data, features, target, task_type, random_state=42, max_rows=3000):
    working_data = data[features + [target]].dropna(subset=[target]).copy()
    if len(working_data) > max_rows:
        working_data = working_data.sample(max_rows, random_state=random_state)

    X = working_data[features]
    y = working_data[target]

    numeric_features = X.select_dtypes(include="number").columns.tolist()
    categorical_features = X.select_dtypes(exclude="number").columns.tolist()

    transformers = []
    if numeric_features:
        numeric_steps = Pipeline([
            ("fill_missing", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ])
        transformers.append(("numbers", numeric_steps, numeric_features))

    if categorical_features:
        categorical_steps = Pipeline([
            ("fill_missing", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ])
        transformers.append(("categories", categorical_steps, categorical_features))

    preprocessor = ColumnTransformer(transformers=transformers)

    if task_type == "regression":
        neural_network = MLPRegressor(
            hidden_layer_sizes=(16,),
            activation="relu",
            max_iter=800,
            early_stopping=True,
            random_state=random_state,
        )
        model = TransformedTargetRegressor(
            regressor=neural_network,
            transformer=StandardScaler(),
        )
        stratify = None
    elif task_type == "classification":
        model = MLPClassifier(
            hidden_layer_sizes=(16,),
            activation="relu",
            max_iter=600,
            random_state=random_state,
        )
        stratify = y
    else:
        raise ValueError("task_type must be 'regression' or 'classification'")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=random_state, stratify=stratify
    )

    pipeline = Pipeline([
        ("prepare", preprocessor),
        ("model", model),
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    print(f"Target: {target}")
    print(f"Task: {task_type}")
    print(f"Features: {features}")
    print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")

    if task_type == "regression":
        print(f"Mean absolute error: {mean_absolute_error(y_test, predictions):,.1f}")
        print(f"R² score: {r2_score(y_test, predictions):.3f}")
        preview = pd.DataFrame({"actual": y_test.iloc[:8], "predicted": np.round(predictions[:8], 0)})
    else:
        print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
        print(f"Balanced accuracy: {balanced_accuracy_score(y_test, predictions):.3f}")
        preview = pd.DataFrame({"actual": y_test.iloc[:8], "predicted": predictions[:8]})

    print("\nA few held-out predictions:")
    print(preview.to_string(index=False))
    return pipeline


## Dataset 1: House rent

**Prediction question:** What monthly rent should we expect for a listed home?

This Kaggle dataset contains more than 4,700 rental listings from several Indian cities. Run the inspection cell and look at the column names, data types, and target values before choosing a model.


In [ ]:
print(apartment_data.head(10).to_string(index=False))
print("\nColumn types:")
print(apartment_data.dtypes)


### Your choices

1. The prediction question points to which **target** column?
2. Does that target contain a measured number or a category?
3. Which columns give simple information about size, rooms, location, and furnishing?
4. Which columns could we leave out of a first model because they need extra parsing or contain many unique labels?

Write your best choices in the workspace below. If you get stuck, all guided answers are collected at the very bottom of the notebook.


In [ ]:
# Student workspace: Dataset 1
# Uncomment the lines and replace the ... placeholders.
# target_1 = "..."
# task_1 = "..."
# features_1 = ["...", "..."]
# rent_model = run_guided_model(apartment_data, features_1, target_1, task_1)


## Dataset 2: Customer churn

**Prediction question:** Will a telecom customer leave the company?

The Kaggle table contains customer services, account history, and billing information. Print the data first. The target count shows the possible outcomes and whether one class is more common.


In [ ]:
print(renewal_data.head(10).to_string(index=False))
print("\nColumn types:")
print(renewal_data.dtypes)


### Your choices

1. Which column contains the outcome we want?
2. Does the target represent an amount or a named group?
3. Which ID column should stay out of the model?
4. Which account, billing, contract, and service columns are known before churn happens?

Write your best choices in the workspace below. The guided version is available at the very bottom if you need it.


In [ ]:
# Student workspace: Dataset 2
# Uncomment the lines and replace the ... placeholders.
# target_2 = "..."
# task_2 = "..."
# features_2 = ["...", "..."]
# churn_model = run_guided_model(renewal_data, features_2, target_2, task_2)


## Dataset 3: Iris species

**Prediction question:** Which iris species produced these flower measurements?

The classic Iris dataset contains 150 flowers from three species. Each flower has four measurements and one row ID.


In [ ]:
print(iris_data.head(10).to_string(index=False))
print("\nColumn types:")
print(iris_data.dtypes)


### Your choices

1. Which target answers the species question?
2. The target has three species names. Which task type handles that?
3. Which column only identifies a row?
4. Which four measurements describe the flower before prediction?

Write your best choices in the workspace below before checking the guided version at the bottom.


In [ ]:
# Student workspace: Dataset 3
# Uncomment the lines and replace the ... placeholders.
# target_3 = "..."
# task_3 = "..."
# features_3 = ["...", "..."]
# iris_model = run_guided_model(iris_data, features_3, target_3, task_3)


## Quick self-check

1. A target contains house prices. Regression or classification?
2. A target contains "Spam" and "Not spam". Regression or classification?
3. Why should a row ID usually stay out of the features?
4. Why is information created after the outcome dangerous?
5. What are the target and feature columns in one dataset from this notebook?

Try answering from memory. The answer key is included with the guided help at the very bottom.


## Finish line

You can now look at a new table and make three core modeling choices:

- name the target,
- distinguish regression from classification,
- choose features that are useful and available before prediction.

Those choices matter before neural-network architecture, hidden-layer size, or any other tuning decision.


---

## Need help? Guided answers

Stop here if you are still working through the datasets. The cells below contain complete starting choices and runnable models for all three activities.


In [ ]:
# Guided answer for Dataset 1
target_1 = "Rent"
task_1 = "regression"
features_1 = [
    "BHK",
    "Size",
    "City",
    "Furnishing Status",
    "Bathroom",
    "Area Type",
]

rent_model = run_guided_model(apartment_data, features_1, target_1, task_1)


<details>
<summary><strong>Why these choices?</strong></summary>

- "Rent" is the target because it answers the prediction question.
- Rent is a measured number, so this is regression.
- BHK, size, city, furnishing, bathrooms, and area type describe the listing before rent prediction.
- We leave out posting date, floor text, locality, tenant preference, and point of contact to keep the first model understandable.

</details>


In [ ]:
# Guided answer for Dataset 2
target_2 = "Churn"
task_2 = "classification"
features_2 = [
    "tenure",
    "MonthlyCharges",
    "Contract",
    "InternetService",
    "TechSupport",
    "PaperlessBilling",
]

churn_model = run_guided_model(renewal_data, features_2, target_2, task_2)


<details>
<summary><strong>Why these choices?</strong></summary>

- "Churn" is the target because it contains the Yes/No outcome.
- Yes and No are categories, so this is classification.
- "customerID" names a record rather than explaining churn.
- Tenure, charges, contract type, internet service, technical support, and billing preference exist before the customer leaves.

</details>


In [ ]:
# Guided answer for Dataset 3
target_3 = "Species"
task_3 = "classification"
features_3 = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm",
]

iris_model = run_guided_model(iris_data, features_3, target_3, task_3, max_rows=1000)


<details>
<summary><strong>Why these choices?</strong></summary>

- "Species" is the target because it contains the flower class.
- The three species names make this a classification task.
- "Id" only identifies a row.
- Sepal length, sepal width, petal length, and petal width are the measured features.

</details>


### Compare the guided decisions

| Kaggle dataset | Target | Task | Features excluded on purpose |
|---|---|---|---|
| House rent | "Rent" | Regression | Complicated text columns for the first model |
| Telco churn | "Churn" | Classification | "customerID" |
| Iris species | "Species" | Classification | "Id" |

The neural-network library changes the output behavior for regression and classification, but the reasoning starts before the model: name the target and choose information that would truly be available.


## Small experiment

Choose one experiment. Change only one thing, rerun the model cell, and compare the score.

- Remove "Size" from the rent features.
- Remove "Contract" from the churn features.
- Remove both petal measurements from the iris features.

Write two sentences: What changed? Why do you think it changed?


### Quick self-check answers

1. Regression, because price is a measured number.
2. Classification, because the target contains named groups.
3. An ID usually identifies the row without describing the situation we want to predict.
4. It leaks future information that would not exist when the real prediction happens.
5. Answers vary by dataset. The target is the outcome. Features are the inputs available before that outcome.
